In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import seaborn as sns
from scipy import stats

ox.settings.use_cache = True

In [ ]:
# FIRST, inspect a few non-street networks to compare to our street networks below
# calculate edge-normalized circuit rank
# graph needs to be undirected for circuit rank to make sense
# if we don't know `c`, approximate it as 1 (since its irrelevant for sufficiently large graphs)
def encr(n, m, c=1):
    cr = m - n + c
    ratio = cr / m
    return round(ratio, 3)


# msn messenger 2006 from: https://arxiv.org/abs/0803.0939
# "180 million nodes and 1.3 billion undirected edges"
print("MSN", encr(n=180_000_000, m=1_300_000_000))

# European airline network (undirected) from https://icon.colorado.edu/
# https://www.nature.com/srep/2013/130227/srep01344/full/srep01344.html
print("EAN", encr(n=450, m=3588))

# paris subway network (undirected)
# from: world subways 2009 https://icon.colorado.edu/networks
# https://royalsocietypublishing.org/doi/full/10.1098/rsif.2012.0259
print("Paris", encr(n=299, m=712))

# delhi bus route network (undirected)
# from: india bus routes (2016) https://icon.colorado.edu/networks
# https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0168478
print("Delhi", encr(n=1554, m=4285))

In [ ]:
# NEXT, load our street networks and compare their values
df_ind = pd.read_csv("data/indicators-all.csv")
df_ind.shape

In [ ]:
df_mst = pd.read_csv("data/mst-results_no_treewidth-neat.csv")
df_mst["uc_id"] = df_mst["id"].str.split("-", expand=True)[1].astype(int)
df_mst = df_mst.drop(columns=["id"])
df_mst.shape

In [ ]:
df = df_ind.merge(df_mst, on="uc_id", how="inner")
df["intersect_density"] = df["intersect_count"] / df["built_up_area_m2"]


# calculate meshedness coefficient, aka alpha index, which is
# the ratio of the number of cycles in the graph to the
# maximum possible number of cycles in a planar graph with the
# same number of nodes
def meshedness(n, m, c=1):
    return (m - n + c) / (2 * n - 5)


df["meshedness"] = meshedness(n=df["node_count"], m=df["edge_count_orig"])

In [ ]:
df = df[df["node_count"] > 100]
df.shape

In [ ]:
df_mst

In [ ]:
# whole graph vs connected component makes little difference to these measures
cols = [
    "edge_count_ratio",
    "length_ratio",
    "edge_count_ratio_cc",
    "length_ratio_cc",
    "meshedness",
    "cc_length_ratio",
    "treewidth",
    "treewidth_cc",
]
df[cols].describe()

## Examine the extent to which MST approximates the full network

length ratio = length of MST / length of full network

edge count ratio = edge count in MST / edge count in full network

In [ ]:
sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(5, 5))
bins = 40
alpha = 0.6
ax = df["length_ratio"].hist(bins=bins, alpha=alpha, ax=ax, color="#003399", label="Length Ratio")
ax = df["edge_count_ratio"].hist(
    bins=bins, alpha=alpha, ax=ax, color="#cc0000", label="Edge Count Ratio"
)
ax.set_xlim(0, 1)
_ = plt.legend()
fig.savefig("figures/histograms-lr-ecr.png", dpi=300, bbox_inches="tight")

## Same thing again, but for the opposite

Instead of how closely MST approximates full network, look at how much "extra stuff" the full network has compared to the MST

Circuit rank ratio (edge-normlized circuit rank) is how many links need to be removed from full network to get the MST. That is, how many links are redundant? (Measure of how non-tree-like, aka how structurally redundant, aka what share of edges form cycles instead of uniquely connect nodes?)

Also calculate length redundancy: what share of full network's length is in excess of the MST?

Both of these indicators range from 0 to 1, where 0 is a tree and 1 is less tree-like.

In [ ]:
# edge_count_ratio + circuit_rank_ratio should equal 1, because...
# circuit_rank_ratio = (edge_count - node_count + component_count) / edge_count
# edge_count_ratio = edge_count_mst / edge_count_original = 1 - circuit_rank_ratio
(df["edge_count_ratio"] + df["circuit_rank_ratio"]).describe().round(4)

In [ ]:
df["length_redundancy"] = 1 - df["length_ratio"]

In [ ]:
# describe edge-normalized circuit ranks for our street networks
# typical is ~28%
# compare this to the other network types analyzed at top of notebook
df_ind = df[["circuit_rank_ratio", "length_redundancy"]]
table = df_ind.describe().round(3)
print(table.to_latex(float_format="%.3f"))
table

In [ ]:
df_ind.std() / df_ind.mean()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))
bins = 40
alpha = 0.6
ax = df["length_redundancy"].hist(
    bins=bins,
    density=False,
    alpha=alpha,
    ax=ax,
    color="#003399",
    label="Length Redundancy ($\\lambda$)",
)
ax = df["circuit_rank_ratio"].hist(
    bins=bins,
    density=False,
    alpha=alpha,
    ax=ax,
    color="#cc0000",
    label="Link Redundancy ($\\theta$)",
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 800)
ax.set_ylabel("Observations")
_ = plt.legend(fontsize=9)
# fig.tight_layout()
fig.savefig("figures/fig_histogram_indicators.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
bins = 40
alpha = 0.6
ax = df["circuit_rank_ratio"].hist(
    bins=bins, density=False, alpha=alpha, ax=ax, color="#cc0000", label="Link Redundancy"
)
ax.set_xlim(0, 1)
_ = plt.legend()
fig.savefig("figures/histogram-cr-alone.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
bins = 100
alpha = 0.6
ax = df["treewidth"].hist(bins=bins, density=False, alpha=alpha, ax=ax, color="#003399")
ax.set_xlabel("Treewidth")
ax.set_xlim(0, 120)
_ = ax.set_ylim(0, 1800)

In [ ]:
df_theta = df.groupby("world_region")["circuit_rank_ratio"].agg(
    Min="min", Mean="mean", Median=np.median, Max="max"
)
df_theta.columns = pd.MultiIndex.from_product([["theta"], df_theta.columns])
df_lambda = df.groupby("world_region")["length_redundancy"].agg(
    Min="min", Mean="mean", Median=np.median, Max="max"
)
df_lambda.columns = pd.MultiIndex.from_product([["lambda"], df_lambda.columns])
df_table = pd.concat([df_theta, df_lambda], axis=1).sort_index().round(3)
print(df_table.to_latex(float_format="%.3f"))
df_table

In [ ]:
cols = ["circuit_rank_ratio", "length_redundancy"]
df.groupby("world_region")[cols].mean().sort_values("circuit_rank_ratio").round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = 30
alpha = 0.7
for label, group in df.groupby("world_region"):
    ax = group["circuit_rank_ratio"].hist(
        bins=bins, density=False, alpha=alpha, ax=ax, label=label, zorder=-len(group)
    )
ax.set_xlim(0, 0.6)
ax.set_ylim(0, 240)
ax.set_xlabel("Link Redundancy")
_ = plt.legend()

In [ ]:
df["length_total_km"] = df["length_total"] / 1000
df["intersect_density_km2"] = df["intersect_density"] * 1e6
df["built_up_area_km2"] = df["built_up_area_m2"] / 1e6
df["pop_density_km2_1000s"] = (df["resident_pop"] / 1000) / df["built_up_area_km2"]
df["built_up_area_km2_log"] = np.log(df["built_up_area_km2"])
df["length_total_km_log"] = np.log(df["length_total_km"])

In [ ]:
x_col = "pop_density_km2_1000s"
y_col = "circuit_rank_ratio"
r1, p1 = stats.pearsonr(df[x_col], df[y_col])
r2, p2 = stats.pearsonr(np.log(df[x_col]), df[y_col])
print(f"r = {r1:.3f}, p = {p1:.3f}")
print(f"r = {r2:.3f}, p = {p2:.3f}")

In [ ]:
# calculate correlations
y_cols = [
    "length_total_km_log",  # log
    "intersect_density_km2",
    "orientation_entropy",
    "bc_gini",
    "k_avg",
    "elev_median",
    "circuity",
    "elev_std",
    "built_up_area_km2_log",  # log
    "pop_density_km2_1000s",
]
corrs = []
for x_col in ["circuit_rank_ratio", "length_redundancy"]:
    for y_col in y_cols:
        r, p = stats.pearsonr(df[x_col], df[y_col])
        corrs.append((x_col, y_col, r, p))

# show results
cols = ["x_col", "y_col", "r", "p"]
df_corrs = pd.DataFrame(corrs, columns=cols)  # .set_index(["x_col", "y_col"])
df_corrs.sort_values(by="r", key=abs, ascending=False).round(3)

In [ ]:
stats.pearsonr(df["circuit_rank_ratio"], df["length_redundancy"])

In [ ]:
# avg degree, BC Gini, circuity
# intersect density, orientation entropy, total length
# hilliness, population density, built up area
df[y_cols].describe().T.round(2)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 13), facecolor="w")
x_cols = [
    "k_avg",
    "bc_gini",
    "circuity",
    "intersect_density_km2",
    "orientation_entropy",
    "length_total_km_log",
    "elev_std",
    "pop_density_km2_1000s",
    "built_up_area_km2_log",
]
x_labels = [
    "Average Node Degree",
    "Gini of Node Betweenness Centralities",
    "Average Circuity",
    "Intersection Density",
    "Orientation Entropy",
    "Total Network Length (log)",
    "Std Dev of Node Elevations",
    "Population Density",
    "Built-up Area (log)",
]
y_cols = ["circuit_rank_ratio", "length_redundancy"]
y_labels = ["Link Redundancy ($\\theta$)", "Length Redundancy ($\\lambda$)"]
xlims = [
    (1.9, 3.7),
    (0.4, 0.95),
    (1, 1.1),
    (0, 800),
    (2.4, 3.59),
    (3.5, 8),
    (1, 60),
    (5, 100),
    (0, 4.5),
]
colors = ["#cc0000", "#003399"]
for x_col, ax, xlim, x_label in zip(x_cols, axes.flat, xlims, x_labels, strict=False):
    r_label = ""
    for y_col, color, y_label in zip(y_cols, colors, y_labels, strict=False):
        ax.scatter(df[x_col], df[y_col], s=1, alpha=0.03, color=color)

        m, b, r, p, se = stats.linregress(df[x_col], df[y_col])
        ax.axline((0, b), slope=m, color=color, linestyle="--", linewidth=1, label=y_label)

        indicator = "$\\lambda$" if y_col == "length_redundancy" else "$\\theta$"
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        r_label += f"$r$({indicator})={r:.2f}{stars}$~~~$"

    if x_col in {"k_avg", "intersect_density_km2", "elev_std"}:
        ax.set_ylabel("Redundancy")

    ax.set_title(r_label.rstrip())
    ax.set_xlabel(x_label)
    ax.set_xlim(xlim)
    ax.set_ylim(0.1, 0.65)
    ax.set_box_aspect(1)
    if x_col == "k_avg":
        ax.legend()

fig.tight_layout()
fig.savefig("figures/fig_scatter_regressions.png", dpi=300, bbox_inches="tight")

In [ ]:
count = 3  # how many of each?
min_edges = 600
max_edges = 900
cols = [
    "country",
    "country_iso",
    "core_city",
    "uc_id",
    "node_count",
    "edge_count_ratio_cc",
    "length_ratio_cc",
]
mask = (df["edge_count_orig_cc"] > min_edges) & (df["edge_count_orig_cc"] < max_edges)
df_subset = df[mask][cols].sort_values("length_ratio_cc")
uc_ids = df_subset["uc_id"].head(count).tolist() + df_subset["uc_id"].tail(count).tolist()
# uc_ids = [1997, 11259]#[7493, 11151]

In [ ]:
df_subset

In [ ]:
%%script false --no-raise-error
bgcolor = "#262626"
fig, axes = plt.subplots(2, count, figsize=(count * 4, 8), facecolor=bgcolor)
for uc_id, ax in zip(uc_ids, axes.flat, strict=False):
    row = df[df["uc_id"] == uc_id].iloc[0]
    root = Path("/data/snm/models/graphml")
    fp = (
        root
        / f"{row['country']}-{row['country_iso']}"
        / f"{row['core_city']}-{row['uc_id']}.graphml"
    )
    Gu = ox.convert.to_undirected(ox.truncate.largest_component(ox.io.load_graphml(fp)))
    mst = nx.minimum_spanning_tree(Gu, weight="length")
    print(row["length_ratio"])
    ax.set_facecolor(bgcolor)
    fig, ax = ox.plot.plot_graph(
        Gu, node_size=0, ax=ax, edge_color="r", bgcolor=bgcolor, show=False, close=False
    )
    fig, ax = ox.plot.plot_graph(
        mst,
        node_size=0,
        ax=ax,
        edge_linewidth=1.5,
        bgcolor=bgcolor,
        edge_color="w",
        show=False,
        close=False,
    )
fig.savefig("figures/fig_network_examples.png", dpi=200)

In [ ]:
# Treewidth
# https://en.wikipedia.org/wiki/Treewidth
# https://networkx.org/documentation/stable/reference/algorithms/approximation.html#module-networkx.algorithms.approximation.treewidth
# https://math.stackexchange.com/questions/3335762/measures-for-how-tree-like-a-graph-is?rq=1
#
# Circuit Rank
# circuit_rank = edge_count - node_count + component_count
# circuit_rank / edge_count = 1 - edge_count_ratio
# https://en.wikipedia.org/wiki/Circuit_rank
# https://math.stackexchange.com/questions/3335762/measures-for-how-tree-like-a-graph-is?rq=1
# AKA first betti number? https://math.stackexchange.com/questions/2102889/measuring-tree-ness-and-other-graph-properties

# levinson: treeness: https://onlinelibrary-wiley-com.libproxy2.usc.edu/doi/full/10.1111/j.1538-4632.2007.00707.x
# and derived from the above: https://www-sciencedirect-com.libproxy2.usc.edu/science/article/pii/S0965856417302549

# yikes! david used similar language to my idea 10 years ago!
# https://www.transportist.net/p/sometimes-a-city-is-a-tree

In [ ]:
df_big = df[df["resident_pop"] > 10_000_000]
df_big.shape

In [ ]:
cols = ["country", "core_city", "resident_pop", "length_redundancy", "circuit_rank_ratio"]
df_big.sort_values("circuit_rank_ratio")[cols]

In [ ]:
dist = 900
nt = "drive"
graphs = []
points = [(-34.733, -58.282), (13.730, 100.438)]
for point in points:
    G = ox.graph.graph_from_point(point, dist=dist, network_type=nt, truncate_by_edge=True)
    graphs.append(G)

fig, axs = plt.subplots(1, 2, figsize=(10, 5), facecolor="#111111")

for ax, G, width in zip(axs, graphs, [3.5, 2.5], strict=False):
    ax.set_facecolor("#111111")
    ox.plot.plot_figure_ground(G, ax=ax, default_width=width, show=False, close=False)

fig.tight_layout(pad=0)
fig.savefig("figures/fig_ba_vs_bangkok.png", dpi=200)
plt.show()